# Lekcija 17 - Ustvarjanje lokalnih AI agentov s Foundry Local in Qwen

V tej zvezku zgradite **lokalnega inženirskega asistenta**, ki deluje izključno na vaši delovni postaji. Kjerkoli ni uporabljena oblačna inferenca. Asistent lahko:

1. **Kliče orodja** preko Qwen klicanja funkcij skozi Foundry Local.
2. **Našteje in prebere datoteke** znotraj peskovnika projekta.
3. **Analizira kodo** z enostavnimi metrikami.
4. **Išče dokumentacijo** z lokalnim RAG (Chroma).
5. **Uporablja lokalni MCP strežnik** (ob odsotnosti lepo preskočeno).

Koda agenta je skoraj enaka kot pri oblačnih lekcijah — le odjemalska končna točka se premakne iz oblaka na `localhost`.


## Namestitev

Pred zagonom tega zvezka:

1. **Namestite Microsoft Foundry Local** (poglejte [dokumentacijo](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) za vaš operacijski sistem).
2. **Prenesite in zaženite model Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Namestite spodnje Python pakete.

Vse poteka lokalno. Stroj z okoli 8 GB RAM-a je realističen minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Povezava z Foundry Local

`FoundryLocalManager` prenese model, če je to potrebno, zažene lokalno storitev in nam omogoči **končno točko združljivo z OpenAI**. Nato standardni OpenAI SDK usmerimo nanjo. API ključ je lokalni rezervni znak — brez vključenih poverilnic za oblak.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Lokalna orodja (peskovniške datotečne operacije)

Ustvarimo majhen vzorčni projekt na disku, nato pa definiramo orodja, omejena na koren tega projekta. Preverjanje peskovnika je pomembno tudi lokalno: orodje, ki bere poljubne poti, deluje z dovoljenji vašega uporabnika in lahko dostopa do vsega, do česar lahko tudi vi.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokalni RAG s Chroma

V lokalno zbirko Chroma vložimo majhen nabor izsekov dokumentacije. Chroma teče znotraj procesa in shrani vektorje na disk — brez strežnika, brez oblaka. Orodje `search_docs` pridobi najbolj relevantne izsek za poizvedbo.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Zanka klicanja orodij

Zdaj registriramo orodja v modelu z uporabo sheme orodij OpenAI in zaženemo standardno zanko klicanja orodij — model zahteva orodje, ga mi izvedemo lokalno, posredujemo rezultat nazaj in ponavljamo, dokler model ne poda končnega odgovora. Zanesljivo klicanje funkcij Qwen je tisto, kar omogoča, da to deluje na napravi.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokalni MCP (neobvezno)

MCP je transport, ne storitev v oblaku — MCP strežnik lahko teče kot lokalni proces prek `stdio`. Celica spodaj prikazuje, kako se lahko povežete z lokalnim MCP strežnikom, če imate enega nastavljeno (na primer datotečni strežnik). V primeru, da `LOCAL_MCP_COMMAND` ni nastavljena, se elegantno preskoči, tako da zvezek vseeno deluje od začetka do konca.

Varnostno opozorilo: lokalni MCP strežnik teče z dovoljenji vašega uporabnika. Omejite ga na imenik projekta in preverite njegove izhode, preden ukrepate na njih.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Povzetek

Zgradili ste inženirskega asistenta, ki deluje popolnoma na vašem stroju:

- **Foundry Local** je poganjal model **Qwen** za OpenAI-kompatibilno končno točko — tako da se koda agenta ujema z lekcijami iz oblaka.
- **Orodja v peskovniku** so agentu omogočila dostop do datotek in analizo kode brez zapuščanja imenika projekta.
- **Chroma** je omogočila **lokalni RAG** nad dokumentacijo.
- **Lokalni MCP** je pokazal, kako znova uporabiti MCP ekosistem brez povezave.

Niti enkrat ni bila uporabljena oblačna inferenca.

### Izziv

Razširite lokalnega agenta, da:

1. **Deluje z več MCP strežniki** — povežite strežnik za datotečni sistem in git strežnik ter pustite agentu, da izbere med njima.
2. **Uporablja lokalni pomnilnik** — trajno shrani kratek zapis pogovora na disk, da si pomočnik zapomni prejšnje korake tudi po ponovnem zagonu zvezka.
3. **Podpira lokalno večagentno orkestracijo** — dodajte drugega lokalnega agenta (npr. recenzenta) in naj se oba skupaj lotita naloge.

V naslednji lekciji se boste naučili, kako zavarovati nameščene AI agente.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
